In [7]:
import torch
import torch.nn as nn

class CNNModel(nn.Module):
    def __init__(self, input_channels, D):
        super(CNNModel, self).__init__()
        self.layer1 = nn.BatchNorm1d(input_channels)
        self.layer2 = nn.Conv1d(input_channels, D, kernel_size=3, padding=1)
        self.layer3 = nn.ReLU()
        self.layer4 = nn.BatchNorm1d(D)
        self.layer5 = nn.MaxPool1d(2)
        self.layer6 = nn.Conv1d(D, 2 * D, kernel_size=3, padding=1)
        self.layer7 = nn.ReLU()
        self.layer8 = nn.BatchNorm1d(2 * D)
        self.layer9 = nn.MaxPool1d(2)
        self.layer10 = nn.Conv1d(2 * D, 4 * D, kernel_size=3, padding=1)
        self.layer11 = nn.ReLU()
        self.layer12 = nn.BatchNorm1d(4 * D)
    def forward(self, x):
        print("Input:", x.shape)
        x = self.layer1(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer2(x)
        print("After Conv1d (input -> D):", x.shape)
        x = self.layer3(x)
        print("After ReLU:", x.shape)
        x = self.layer4(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer5(x)
        print("After MaxPool1d(2):", x.shape)
        x = self.layer6(x)
        print("After Conv1d (D -> 2D):", x.shape)
        x = self.layer7(x)
        print("After ReLU:", x.shape)
        x = self.layer8(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer9(x)
        print("After MaxPool1d(2):", x.shape)
        x = self.layer10(x)
        print("After Conv1d (2D -> 4D):", x.shape)
        x = self.layer11(x)
        print("After ReLU:", x.shape)
        x = self.layer12(x)
        print("After BatchNorm1d:", x.shape)
        return x
# Example usage
batch_size = 8
input_channels = 2  # Number of input channels
sequence_length = 18
D = 16
model = CNNModel(input_channels, D)
x = torch.randn(batch_size, input_channels, sequence_length)
output = model(x)

from torchinfo import summary
summary(model, input_size=((8, 2, 18)))


Input: torch.Size([8, 2, 18])
After BatchNorm1d: torch.Size([8, 2, 18])
After Conv1d (input -> D): torch.Size([8, 16, 18])
After ReLU: torch.Size([8, 16, 18])
After BatchNorm1d: torch.Size([8, 16, 18])
After MaxPool1d(2): torch.Size([8, 16, 9])
After Conv1d (D -> 2D): torch.Size([8, 32, 9])
After ReLU: torch.Size([8, 32, 9])
After BatchNorm1d: torch.Size([8, 32, 9])
After MaxPool1d(2): torch.Size([8, 32, 4])
After Conv1d (2D -> 4D): torch.Size([8, 64, 4])
After ReLU: torch.Size([8, 64, 4])
After BatchNorm1d: torch.Size([8, 64, 4])
Input: torch.Size([8, 2, 18])
After BatchNorm1d: torch.Size([8, 2, 18])
After Conv1d (input -> D): torch.Size([8, 16, 18])
After ReLU: torch.Size([8, 16, 18])
After BatchNorm1d: torch.Size([8, 16, 18])
After MaxPool1d(2): torch.Size([8, 16, 9])
After Conv1d (D -> 2D): torch.Size([8, 32, 9])
After ReLU: torch.Size([8, 32, 9])
After BatchNorm1d: torch.Size([8, 32, 9])
After MaxPool1d(2): torch.Size([8, 32, 4])
After Conv1d (2D -> 4D): torch.Size([8, 64, 4])
Aft

Layer (type:depth-idx)                   Output Shape              Param #
CNNModel                                 [8, 64, 4]                --
├─BatchNorm1d: 1-1                       [8, 2, 18]                4
├─Conv1d: 1-2                            [8, 16, 18]               112
├─ReLU: 1-3                              [8, 16, 18]               --
├─BatchNorm1d: 1-4                       [8, 16, 18]               32
├─MaxPool1d: 1-5                         [8, 16, 9]                --
├─Conv1d: 1-6                            [8, 32, 9]                1,568
├─ReLU: 1-7                              [8, 32, 9]                --
├─BatchNorm1d: 1-8                       [8, 32, 9]                64
├─MaxPool1d: 1-9                         [8, 32, 4]                --
├─Conv1d: 1-10                           [8, 64, 4]                6,208
├─ReLU: 1-11                             [8, 64, 4]                --
├─BatchNorm1d: 1-12                      [8, 64, 4]                128
Total pa

In [24]:
class SmallCNNBranch(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, input_len, D):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.BatchNorm1d(input_len),
            nn.Conv1d(input_len, D, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(D),
            nn.MaxPool1d(2),
            nn.Conv1d(D, 2 * D, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(2 * D),
            nn.MaxPool1d(2),
            nn.Conv1d(2 * D, 4 * D, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(4 * D),
        )
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(4 * D, input_len)

    # ------------------------------------------------------------------------------------------------------------------
    def forward(self, x):
        x = self.cnn(x)
        x = self.global_avg_pool(x).squeeze(-1)  # [B, C]
        x = self.fc(x)
        return x

# Example usage
batch_size = 8
input_channels = 2  # Number of input channels
sequence_length = 18
D = 16
model = SmallCNNBranch(input_channels, D)
x = torch.randn(batch_size, input_channels, sequence_length)
output = model(x)

from torchinfo import summary
summary(model, input_size=((8, 2, 18)))

Layer (type:depth-idx)                   Output Shape              Param #
SmallCNNBranch                           [8, 2]                    --
├─Sequential: 1-1                        [8, 64, 4]                --
│    └─BatchNorm1d: 2-1                  [8, 2, 18]                4
│    └─Conv1d: 2-2                       [8, 16, 18]               112
│    └─ReLU: 2-3                         [8, 16, 18]               --
│    └─BatchNorm1d: 2-4                  [8, 16, 18]               32
│    └─MaxPool1d: 2-5                    [8, 16, 9]                --
│    └─Conv1d: 2-6                       [8, 32, 9]                1,568
│    └─ReLU: 2-7                         [8, 32, 9]                --
│    └─BatchNorm1d: 2-8                  [8, 32, 9]                64
│    └─MaxPool1d: 2-9                    [8, 32, 4]                --
│    └─Conv1d: 2-10                      [8, 64, 4]                6,208
│    └─ReLU: 2-11                        [8, 64, 4]                --
│    └─Ba

In [28]:
class SmallCNNBranch(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, input_channels, D):
        super().__init__()
        self.layer1 = nn.BatchNorm1d(input_channels)
        self.layer2 = nn.Conv1d(input_channels, D, kernel_size=3, padding=1)
        self.layer3 = nn.ReLU()
        self.layer4 = nn.BatchNorm1d(D)
        self.layer5 = nn.MaxPool1d(2)
        self.layer6 = nn.Conv1d(D, 2 * D, kernel_size=3, padding=1)
        self.layer7 = nn.ReLU()
        self.layer8 = nn.BatchNorm1d(2 * D)
        self.layer9 = nn.MaxPool1d(2)
        self.layer10 = nn.Conv1d(2 * D, 4 * D, kernel_size=3, padding=1)
        self.layer11 = nn.ReLU()
        self.layer12 = nn.BatchNorm1d(4 * D)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(4 * D, input_channels)

    def forward(self, x):
        print("Input:", x.shape)
        x = self.layer1(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer2(x)
        print("After Conv1d (input -> D):", x.shape)
        x = self.layer3(x)
        print("After ReLU:", x.shape)
        x = self.layer4(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer5(x)
        print("After MaxPool1d(2):", x.shape)
        x = self.layer6(x)
        print("After Conv1d (D -> 2D):", x.shape)
        x = self.layer7(x)
        print("After ReLU:", x.shape)
        x = self.layer8(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer9(x)
        print("After MaxPool1d(2):", x.shape)
        x = self.layer10(x)
        print("After Conv1d (2D -> 4D):", x.shape)
        x = self.layer11(x)
        print("After ReLU:", x.shape)
        x = self.layer12(x)
        print("After BatchNorm1d:", x.shape)
        x = self.global_avg_pool(x).squeeze(-1)
        print("After global_avg_pool:", x.shape)
        x = self.fc(x)
        print("After linear:", x.shape)
        return x

# Example usage
batch_size = 8
input_channels = 2  # Number of input channels
sequence_length = 18
D = 16
model = SmallCNNBranch(input_channels, D)
x = torch.randn(batch_size, input_channels, sequence_length)
output = model(x)

from torchinfo import summary
summary(model, input_size=((8, 2, 18)))

Input: torch.Size([8, 2, 18])
After BatchNorm1d: torch.Size([8, 2, 18])
After Conv1d (input -> D): torch.Size([8, 16, 18])
After ReLU: torch.Size([8, 16, 18])
After BatchNorm1d: torch.Size([8, 16, 18])
After MaxPool1d(2): torch.Size([8, 16, 9])
After Conv1d (D -> 2D): torch.Size([8, 32, 9])
After ReLU: torch.Size([8, 32, 9])
After BatchNorm1d: torch.Size([8, 32, 9])
After MaxPool1d(2): torch.Size([8, 32, 4])
After Conv1d (2D -> 4D): torch.Size([8, 64, 4])
After ReLU: torch.Size([8, 64, 4])
After BatchNorm1d: torch.Size([8, 64, 4])
After global_avg_pool: torch.Size([8, 64])
After linear: torch.Size([8, 2])
Input: torch.Size([8, 2, 18])
After BatchNorm1d: torch.Size([8, 2, 18])
After Conv1d (input -> D): torch.Size([8, 16, 18])
After ReLU: torch.Size([8, 16, 18])
After BatchNorm1d: torch.Size([8, 16, 18])
After MaxPool1d(2): torch.Size([8, 16, 9])
After Conv1d (D -> 2D): torch.Size([8, 32, 9])
After ReLU: torch.Size([8, 32, 9])
After BatchNorm1d: torch.Size([8, 32, 9])
After MaxPool1d(2)

Layer (type:depth-idx)                   Output Shape              Param #
SmallCNNBranch                           [8, 2]                    --
├─BatchNorm1d: 1-1                       [8, 2, 18]                4
├─Conv1d: 1-2                            [8, 16, 18]               112
├─ReLU: 1-3                              [8, 16, 18]               --
├─BatchNorm1d: 1-4                       [8, 16, 18]               32
├─MaxPool1d: 1-5                         [8, 16, 9]                --
├─Conv1d: 1-6                            [8, 32, 9]                1,568
├─ReLU: 1-7                              [8, 32, 9]                --
├─BatchNorm1d: 1-8                       [8, 32, 9]                64
├─MaxPool1d: 1-9                         [8, 32, 4]                --
├─Conv1d: 1-10                           [8, 64, 4]                6,208
├─ReLU: 1-11                             [8, 64, 4]                --
├─BatchNorm1d: 1-12                      [8, 64, 4]                128
├─Adapti

In [38]:
class SmallCNNBranch(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, input_channels, D):
        super().__init__()
        self.layer1 = nn.BatchNorm2d(input_channels)
        self.layer2 = nn.Conv2d(input_channels, D, kernel_size=3, padding=0)
        self.layer3 = nn.ReLU()
        self.layer4 = nn.BatchNorm2d(D)
        self.layer5 = nn.MaxPool2d(2)
        self.layer6 = nn.Conv2d(D, 2 * D, kernel_size=3, padding=1)
        self.layer7 = nn.ReLU()
        self.layer8 = nn.BatchNorm2d(2 * D)
        self.layer9 = nn.MaxPool2d(2)
        self.layer10 = nn.Conv2d(2 * D, 4 * D, kernel_size=3, padding=1)
        self.layer11 = nn.ReLU()
        self.layer12 = nn.BatchNorm2d(4 * D)
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(4 * D, 2 * D)
        self.fc2 = nn.Linear(2 * D, D)
        self.fc3 = nn.Linear(D, input_channels)


    def forward(self, x):
        print("Input:", x.shape)
        x = self.layer1(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer2(x)
        print("After Conv1d (input -> D):", x.shape)
        x = self.layer3(x)
        print("After ReLU:", x.shape)
        x = self.layer4(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer5(x)
        print("After MaxPool1d(2):", x.shape)
        x = self.layer6(x)
        print("After Conv1d (D -> 2D):", x.shape)
        x = self.layer7(x)
        print("After ReLU:", x.shape)
        x = self.layer8(x)
        print("After BatchNorm1d:", x.shape)
        x = self.layer9(x)
        print("After MaxPool1d(2):", x.shape)
        x = self.layer10(x)
        print("After Conv1d (2D -> 4D):", x.shape)
        x = self.layer11(x)
        print("After ReLU:", x.shape)
        x = self.layer12(x)
        print("After BatchNorm1d:", x.shape)
        x = self.global_avg_pool(x).squeeze(-1).squeeze(-1)
        print("After global_avg_pool:", x.shape)
        x = self.fc1(x)
        print("After linear1:", x.shape)
        x = self.fc2(x)
        print("After linear2:", x.shape)
        x = self.fc3(x)
        print("After linear3:", x.shape)
        return x

# Example usage
batch_size = 8
input_channels = 2  # Number of input channels
sequence_length = 18
D = 16
model = SmallCNNBranch(input_channels, D)
x = torch.randn(batch_size, input_channels, sequence_length, 33)
output = model(x)

from torchinfo import summary
summary(model, input_size=((8, 2, 18, 33)))

Input: torch.Size([8, 2, 18, 33])
After BatchNorm1d: torch.Size([8, 2, 18, 33])
After Conv1d (input -> D): torch.Size([8, 16, 16, 31])
After ReLU: torch.Size([8, 16, 16, 31])
After BatchNorm1d: torch.Size([8, 16, 16, 31])
After MaxPool1d(2): torch.Size([8, 16, 8, 15])
After Conv1d (D -> 2D): torch.Size([8, 32, 8, 15])
After ReLU: torch.Size([8, 32, 8, 15])
After BatchNorm1d: torch.Size([8, 32, 8, 15])
After MaxPool1d(2): torch.Size([8, 32, 4, 7])
After Conv1d (2D -> 4D): torch.Size([8, 64, 4, 7])
After ReLU: torch.Size([8, 64, 4, 7])
After BatchNorm1d: torch.Size([8, 64, 4, 7])
After global_avg_pool: torch.Size([8, 64])
After linear1: torch.Size([8, 32])
After linear2: torch.Size([8, 16])
After linear3: torch.Size([8, 2])
Input: torch.Size([8, 2, 18, 33])
After BatchNorm1d: torch.Size([8, 2, 18, 33])
After Conv1d (input -> D): torch.Size([8, 16, 16, 31])
After ReLU: torch.Size([8, 16, 16, 31])
After BatchNorm1d: torch.Size([8, 16, 16, 31])
After MaxPool1d(2): torch.Size([8, 16, 8, 15])

Layer (type:depth-idx)                   Output Shape              Param #
SmallCNNBranch                           [8, 2]                    --
├─BatchNorm2d: 1-1                       [8, 2, 18, 33]            4
├─Conv2d: 1-2                            [8, 16, 16, 31]           304
├─ReLU: 1-3                              [8, 16, 16, 31]           --
├─BatchNorm2d: 1-4                       [8, 16, 16, 31]           32
├─MaxPool2d: 1-5                         [8, 16, 8, 15]            --
├─Conv2d: 1-6                            [8, 32, 8, 15]            4,640
├─ReLU: 1-7                              [8, 32, 8, 15]            --
├─BatchNorm2d: 1-8                       [8, 32, 8, 15]            64
├─MaxPool2d: 1-9                         [8, 32, 4, 7]             --
├─Conv2d: 1-10                           [8, 64, 4, 7]             18,496
├─ReLU: 1-11                             [8, 64, 4, 7]             --
├─BatchNorm2d: 1-12                      [8, 64, 4, 7]             128
├─Adapt

In [22]:
class SmallCNNBranch(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, input_len, D):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.BatchNorm2d(input_len),
            nn.Conv2d(input_len, D, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(D),
            nn.MaxPool2d(2),
            # nn.Conv1d(D, 2 * D, kernel_size=3, padding=1),
            # nn.ReLU(),
            # nn.BatchNorm2d(2 * D),
            # nn.MaxPool2d(2),
            # nn.Conv2d(2 * D, 4 * D, kernel_size=3, padding=1),
            # nn.ReLU(),
            # nn.BatchNorm2d(4 * D),
        )
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(4 * D, input_len)

    # ------------------------------------------------------------------------------------------------------------------
    def forward(self, x):
        x = self.cnn(x)
        print("After CNN:", x.shape)
        x = self.global_avg_pool(x).squeeze(-1)  # [B, C]
        print("After global_avg_pool:", x.shape)
        x = self.fc(x)
        print("After Linear:", x.shape)
        return x

# Example usage
batch_size = 8
input_channels = 2  # Number of input channels
sequence_length = 18
D = 16
model = SmallCNNBranch(input_channels, D)
x = torch.randn(batch_size, input_channels, sequence_length, 13)
output = model(x)

from torchinfo import summary
summary(model, input_size=((8, 2, 18, 13)))

After CNN: torch.Size([8, 16, 9, 6])


RuntimeError: Expected 2 to 3 dimensions, but got 4-dimensional tensor for argument #1 'self' (while checking arguments for adaptive_avg_pool1d)